# 🐍 Обучение змеи с помощью генетического алгоритма

Этот ноутбук позволяет обучить нейронную сеть играть в змейку, используя генетический алгоритм.

**Автор оригинального проекта:** Valentin Macé  
**Репозиторий:** https://github.com/valentinmace/snake

## Как это работает:
1. Создается популяция нейронных сетей со случайными весами
2. Каждая сеть играет в змейку и получает оценку (fitness)
3. Лучшие сети отбираются как родители
4. Создаются новые сети через скрещивание и мутацию
5. Процесс повторяется много поколений

## Параметры обучения:
- **Архитектура сети:** [21, 16, 3] - 21 вход (зрение змеи), 16 скрытых нейронов, 3 выхода (прямо/лево/право)
- **Размер популяции:** 1000 сетей
- **Метод кроссовера:** 'neuron' - обмен нейронами между родителями
- **Метод мутации:** 'weight' - изменение отдельных весов

## 📦 Шаг 1: Установка зависимостей и клонирование репозитория

In [ ]:
# Клонируем репозиторий
!git clone https://github.com/valentinmace/snake.git
%cd snake

# Устанавливаем необходимые библиотеки
!pip install numpy numba joblib -q

print("✅ Зависимости установлены!")

## 🔧 Шаг 2: Адаптация кода для Colab (без pygame)

In [ ]:
# Создаем модифицированную версию neural_network.py без pygame
neural_network_code = '''
# Модифицированная версия для Colab (без pygame)
from numba import jit
import numpy as np

class NeuralNetwork:
    """Neural Network class"""

    def __init__(self, shape=None):
        self.shape = shape
        self.biases = []
        self.weights = []
        self.score = 0
        if shape:
            for y in shape[1:]:
                self.biases.append(np.random.randn(y, 1))
            for x, y in zip(shape[:-1], shape[1:]):
                self.weights.append(np.random.randn(y, x))

    def feed_forward(self, a):
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)
        return a

    def save(self, name=None):
        if not name:
            np.save('saved_weights_'+str(self.score), self.weights)
            np.save('saved_biases_'+str(self.score), self.biases)
        else:
            np.save(name + '_weights', self.weights)
            np.save(name + '_biases', self.biases)

    def load(self, filename_weights, filename_biases):
        self.weights = np.load(filename_weights, allow_pickle=True)
        self.biases = np.load(filename_biases, allow_pickle=True)

    def render(self, window, vision):
        pass  # Не используется в режиме обучения

@jit(nopython=True)
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))
'''

with open('neural_network_colab.py', 'w') as f:
    f.write(neural_network_code)

# Создаем модифицированную версию game.py без pygame
game_code = '''
# Модифицированная версия для Colab (только невидимый режим)
from map import *
from snake import *

class Game:
    def __init__(self):
        self.game_score = 0
        self.game_time = 0

    def start(self, display=False, neural_net=None, playable=False, speed=20):
        return self.run_invisible(neural_net=neural_net)

    def run_invisible(self, neural_net=None):
        snake = Snake(neural_net=neural_net)
        map = Map(snake)

        cont = True
        while cont:
            self.game_time += 1
            map.scan()
            snake.AI()
            snake.update()
            map.update()
            if not snake.alive or self.game_time > 1000:  # Добавлен лимит итераций
                cont = False
                self.game_time = 0
        self.game_score = snake.fitness()
        return self.game_score

    def run_visible(self, playable=False, neural_net=None, speed=20):
        return self.run_invisible(neural_net=neural_net)

    def inputs_management(self, snake, cont):
        pass

    def render(self, window, map):
        pass
'''

with open('game_colab.py', 'w') as f:
    f.write(game_code)

# Создаем модифицированную версию genetic_algorithm.py
ga_code = '''
# Модифицированная версия для Colab
import copy
import multiprocessing
from random import randint
import random
import numpy as np
from game_colab import Game
from neural_network_colab import NeuralNetwork
from joblib import Parallel, delayed

class GeneticAlgorithm:
    def __init__(self, networks=None, networks_shape=None, population_size=1000, generation_number=100,
                 crossover_rate=0.3, crossover_method='neuron', mutation_rate=0.7, mutation_method='weight'):
        self.networks_shape = networks_shape
        if self.networks_shape is None:
            self.networks_shape = [21,16,3]
        self.networks = networks

        if networks is None:
            self.networks = []
            for i in range(population_size):
                self.networks.append(NeuralNetwork(self.networks_shape))

        self.population_size = population_size
        self.generation_number = generation_number
        self.crossover_rate = crossover_rate
        self.crossover_method = crossover_method
        self.mutation_rate = mutation_rate
        self.mutation_method = mutation_method
        self.best_scores = []  # Для отслеживания прогресса

    def start(self):
        networks = self.networks
        population_size = self.population_size
        crossover_number = int(self.crossover_rate*self.population_size)
        mutation_number = int(self.mutation_rate*self.population_size)

        num_cores = multiprocessing.cpu_count()
        gen = 0
        for i in range(self.generation_number):
            gen += 1
            print(f"\n{'='*60}")
            print(f"🧬 ПОКОЛЕНИЕ {gen}/{self.generation_number}")
            print(f"{'='*60}")

            parents = self.parent_selection(networks, crossover_number, population_size)
            children = self.children_production(crossover_number, parents)
            mutations = self.mutation_production(networks, mutation_number, population_size)

            networks = networks + children + mutations
            print(f"⚡ Оценка {len(networks)} нейронных сетей...")
            self.evaluation(networks, num_cores)
            networks.sort(key=lambda Network: Network.score, reverse=True)
            
            # Сохраняем лучшую сеть
            networks[0].save(name=f"gen_{gen}_best")
            self.best_scores.append(networks[0].score)

            for i in range(int(0.2*len(networks))):
                rand = randint(10, len(networks)-1)
                networks[rand] = self.mutation(networks[rand])

            networks = networks[:population_size]
            self.print_generation(networks, gen)

    def parent_selection(self, networks, crossover_number, population_size):
        parents = []
        for i in range(crossover_number):
            parent = self.tournament(networks[randint(0, population_size - 1)],
                                     networks[randint(0, population_size - 1)],
                                     networks[randint(0, population_size - 1)])
            parents.append(parent)
        return parents

    def children_production(self, crossover_number, parents):
        children = []
        for i in range(crossover_number):
            child = self.crossover(parents[randint(0, crossover_number - 1)],
                                   parents[randint(0, crossover_number - 1)])
            children.append(child)
        return children

    def mutation_production(self, networks, mutation_number, population_size):
        mutations = []
        for i in range(mutation_number):
            mut = self.mutation(networks[randint(0, population_size - 1)])
            mutations.append(mut)
        return mutations

    def evaluation(self, networks, num_cores):
        game = Game()
        results1 = Parallel(n_jobs=num_cores)(delayed(game.start)(neural_net=networks[i]) for i in range(len(networks)))
        results2 = Parallel(n_jobs=num_cores)(delayed(game.start)(neural_net=networks[i]) for i in range(len(networks)))
        results3 = Parallel(n_jobs=num_cores)(delayed(game.start)(neural_net=networks[i]) for i in range(len(networks)))
        results4 = Parallel(n_jobs=num_cores)(delayed(game.start)(neural_net=networks[i]) for i in range(len(networks)))
        for i in range(len(results1)):
            networks[i].score = int(np.mean([results1[i], results2[i], results3[i], results4[i]]))

    def tournament(self, net1, net2, net3):
        game = Game()
        game.start(neural_net=net1)
        score1 = game.game_score
        game.start(neural_net=net2)
        score2 = game.game_score
        game.start(neural_net=net3)
        score3 = game.game_score
        maxscore = max(score1, score2, score3)
        if maxscore == score1:
            return net1
        elif maxscore == score2:
            return net2
        else:
            return net3

    def crossover(self, net1, net2):
        res1 = copy.deepcopy(net1)
        res2 = copy.deepcopy(net2)
        weights_or_biases = random.randint(0, 1)
        if weights_or_biases == 0:
            if self.crossover_method == 'weight':
                layer = random.randint(0, len(res1.weights) - 1)
                neuron = random.randint(0, len(res1.weights[layer]) - 1)
                weight = random.randint(0, len(res1.weights[layer][neuron]) - 1)
                temp = res1.weights[layer][neuron][weight]
                res1.weights[layer][neuron][weight] = res2.weights[layer][neuron][weight]
                res2.weights[layer][neuron][weight] = temp
            elif self.crossover_method == 'neuron':
                layer = random.randint(0, len(res1.weights) - 1)
                neuron = random.randint(0, len(res1.weights[layer]) - 1)
                temp = copy.deepcopy(res1)
                res1.weights[layer][neuron] = res2.weights[layer][neuron]
                res2.weights[layer][neuron] = temp.weights[layer][neuron]
            elif self.crossover_method == 'layer':
                layer = random.randint(0, len(res1.weights) - 1)
                temp = copy.deepcopy(res1)
                res1.weights[layer] = res2.weights[layer]
                res2.weights[layer] = temp.weights[layer]
        else:
            layer = random.randint(0, len(res1.biases) - 1)
            bias = random.randint(0, len(res1.biases[layer]) - 1)
            temp = copy.deepcopy(res1)
            res1.biases[layer][bias] = res2.biases[layer][bias]
            res2.biases[layer][bias] = temp.biases[layer][bias]

        game = Game()
        game.start(neural_net=res1)
        score1 = game.game_score
        game.start(neural_net=res2)
        score2 = game.game_score
        if score1 > score2:
            return res1
        else:
            return res2

    def mutation(self, net):
        res = copy.deepcopy(net)
        weights_or_biases = random.randint(0, 1)
        if weights_or_biases == 0:
            if self.mutation_method == 'weight':
                layer = random.randint(0, len(res.weights) - 1)
                neuron = random.randint(0, len(res.weights[layer]) - 1)
                weight = random.randint(0, len(res.weights[layer][neuron]) - 1)
                res.weights[layer][neuron][weight] = np.random.randn()
            elif self.mutation_method == 'neuron':
                layer = random.randint(0, len(res.weights) - 1)
                neuron = random.randint(0, len(res.weights[layer]) - 1)
                new_neuron = np.random.randn(len(res.weights[layer][neuron]))
                res.weights[layer][neuron] = new_neuron
        else:
            layer = random.randint(0, len(res.biases) - 1)
            bias = random.randint(0, len(res.biases[layer]) - 1)
            res.weights[layer][bias] = np.random.randn()
        return res

    def print_generation(self, networks, gen):
        top_mean = int(np.mean([networks[i].score for i in range(6)]))
        bottom_mean = int(np.mean([networks[-i].score for i in range(1, 6)]))
        print(f"\n📊 РЕЗУЛЬТАТЫ ПОКОЛЕНИЯ {gen}:")
        print(f"   🏆 Лучший результат: {networks[0].score}")
        print(f"   👥 Размер популяции: {len(networks)}")
        print(f"   ⭐ Среднее топ-6: {top_mean}")
        print(f"   📉 Среднее худших 6: {bottom_mean}")
'''

with open('genetic_algorithm_colab.py', 'w') as f:
    f.write(ga_code)

print("✅ Модифицированные файлы созданы!")

## 🚀 Шаг 3: Запуск обучения

⚠️ **Внимание:** Обучение может занять несколько часов в зависимости от количества поколений.  
Для тестирования рекомендуется уменьшить `generation_number` до 5-10.

**Параметры:**
- `population_size`: размер популяции (больше = лучше результат, но медленнее)
- `generation_number`: количество поколений
- `crossover_method`: метод скрещивания ('weight', 'neuron', 'layer')
- `mutation_method`: метод мутации ('weight', 'neuron')

In [ ]:
from genetic_algorithm_colab import GeneticAlgorithm
import time

# Настройки обучения
POPULATION_SIZE = 500  # Уменьшено для ускорения в Colab
GENERATIONS = 20       # Можно увеличить для лучших результатов

print("🎮 Начинаем обучение змеи!\n")
print(f"Параметры:")
print(f"  - Размер популяции: {POPULATION_SIZE}")
print(f"  - Количество поколений: {GENERATIONS}")
print(f"  - Архитектура сети: [21, 16, 3]")
print(f"  - Метод кроссовера: neuron")
print(f"  - Метод мутации: weight\n")

start_time = time.time()

# Создаем и запускаем генетический алгоритм
gen = GeneticAlgorithm(
    population_size=POPULATION_SIZE,
    generation_number=GENERATIONS,
    crossover_method='neuron',
    mutation_method='weight'
)

gen.start()

elapsed_time = time.time() - start_time
print(f"\n\n{'='*60}")
print(f"✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"⏱️  Время обучения: {elapsed_time/60:.2f} минут")
print(f"🏆 Лучший результат: {max(gen.best_scores)}")
print(f"{'='*60}")

## 📈 Шаг 4: Визуализация прогресса обучения

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(gen.best_scores, marker='o', linewidth=2, markersize=6)
plt.title('🐍 Прогресс обучения змеи', fontsize=16, fontweight='bold')
plt.xlabel('Поколение', fontsize=12)
plt.ylabel('Лучший результат (fitness)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Статистика обучения:")
print(f"   Начальный результат: {gen.best_scores[0]}")
print(f"   Финальный результат: {gen.best_scores[-1]}")
print(f"   Улучшение: {gen.best_scores[-1] - gen.best_scores[0]} (+{((gen.best_scores[-1] / gen.best_scores[0] - 1) * 100):.1f}%)")
print(f"   Максимальный результат: {max(gen.best_scores)}")

## 💾 Шаг 5: Скачивание обученных моделей

Скачайте лучшие модели для использования локально

In [ ]:
import os
from google.colab import files
import glob

# Находим лучшую модель по номеру поколения
best_gen = gen.best_scores.index(max(gen.best_scores)) + 1
print(f"🏆 Лучшая модель найдена в поколении {best_gen}")
print(f"   Результат: {max(gen.best_scores)}\n")

# Список всех сохраненных файлов
weight_files = sorted(glob.glob('gen_*_weights.npy'))
bias_files = sorted(glob.glob('gen_*_biases.npy'))

print(f"Сохранено {len(weight_files)} моделей\n")

# Скачиваем последнюю (лучшую) модель
if weight_files and bias_files:
    last_weights = weight_files[-1]
    last_biases = bias_files[-1]
    
    print(f"Скачиваем лучшую модель:")
    print(f"  - {last_weights}")
    print(f"  - {last_biases}")
    
    files.download(last_weights)
    files.download(last_biases)
    
    print("\n✅ Файлы успешно скачаны!")
    print("\n📝 Как использовать локально:")
    print("   1. Поместите файлы в папку 'saved/'")
    print("   2. Запустите код:")
    print("      net = NeuralNetwork()")
    print(f"      net.load('saved/{last_weights}', 'saved/{last_biases}')")
    print("      game = Game()")
    print("      game.start(display=True, neural_net=net)")
else:
    print("⚠️ Модели не найдены")

## 🧪 Шаг 6: Тестирование лучшей модели (опционально)

Проверим, насколько хорошо играет лучшая модель

In [ ]:
from neural_network_colab import NeuralNetwork
from game_colab import Game
import numpy as np

# Загружаем лучшую модель
best_net = NeuralNetwork()
best_net.load(f'gen_{GENERATIONS}_best_weights.npy', f'gen_{GENERATIONS}_best_biases.npy')

# Тестируем 10 игр
print("🎮 Тестируем лучшую модель (10 игр):\n")
test_scores = []
game = Game()

for i in range(10):
    score = game.start(neural_net=best_net)
    test_scores.append(score)
    print(f"  Игра {i+1}: {score} очков")

print(f"\n📊 Результаты тестирования:")
print(f"   Средний результат: {np.mean(test_scores):.1f}")
print(f"   Лучший результат: {max(test_scores)}")
print(f"   Худший результат: {min(test_scores)}")
print(f"   Стандартное отклонение: {np.std(test_scores):.1f}")

## 💡 Советы по улучшению результатов

1. **Увеличьте количество поколений** (50-100) для лучших результатов
2. **Экспериментируйте с размером популяции** (500-2000)
3. **Попробуйте разные методы кроссовера и мутации:**
   - crossover_method: 'weight', 'neuron', 'layer'
   - mutation_method: 'weight', 'neuron'
4. **Измените архитектуру сети**, например [21, 20, 12, 3]
5. **Используйте GPU Runtime** в Colab (Runtime → Change runtime type → GPU)
6. **Запустите несколько тренировок** и выберите лучшую модель

## 📚 Дополнительная информация

- Оригинальный репозиторий: https://github.com/valentinmace/snake
- YouTube канал автора: https://www.youtube.com/channel/UCMIW0JKxoxBDM5yiiF17SrA
- Для запуска с визуализацией клонируйте репозиторий и запустите локально с pygame

## 🎯 Как работает система зрения змеи?

Змея "видит" в 8 направлениях (↑↗→↘↓↙←↖) и для каждого получает информацию:
- Расстояние до стены
- Расстояние до еды  
- Расстояние до собственного тела

Всего: 8 направлений × 3 параметра = **24 входа** (но используется 21)

**Выходы (3 нейрона):**
- Повернуть налево
- Идти прямо
- Повернуть направо